In [1]:
import os
import json
import uuid
from typing import TypedDict

from google.generativeai.types import GenerationConfig, datetime
from kscLLM.index import ROOT_PATH
from kscLLM.util import get_model, to_markdown, get_current_time
from kscLLM.models import STIXIndicator

from google.generativeai.generative_models import GenerativeModel
GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY")
print("set") if GOOGLE_API_KEY else print("unset")

set


/home/lukas/Programming/uni/threatintel-showcase/llm/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = get_model()

In [3]:
with open(ROOT_PATH / "tmp/positives.json", "r") as file:
    observables_bundle = json.load(file)

prompt = f"""Create a STIX Indicator matching the the following SDOs.
A credential access attack is searched. First a token is fetched. This token is then extracted by an SSRF attack which calls the metadata.google.internal service accounts token endpoint.
Match the SDOs by using the message property in each artifact.

{str(observables_bundle)}"""

response_indicator = model.generate_content(
    prompt,
    generation_config=GenerationConfig(response_mime_type="application/json", response_schema=STIXIndicator)
)
response: STIXIndicator = json.loads(response_indicator.text)
response

{'description': 'Detects SSRF attempts to fetch Google Cloud service account tokens from the metadata server.',
 'indicator_types': ['malicious-activity'],
 'kill_chain_phases': [{'kill_chain_name': 'mitre-attack',
   'phase_name': 'credential-access'}],
 'name': 'SSRF - Google Cloud Metadata Server Token Fetch',
 'pattern': "([artifact:message MATCHES '.*\\/computeMetadata\\/v1\\/instance\\/service-accounts\\/default\\/token.*'] FOLLOWEDBY [artifact:message MATCHES '.*metadata.google.internal.*'])"}

In [4]:
indicator = {
    "type": "bundle",
    "id": f"bundle--{uuid.uuid4()}",
    "spec_version": "2.1",
    "objects": [
        response | {
            "type": "indicator",
            "spec_version": "2.1",
            "id": f"indicator--{uuid.uuid4()}",
            "created": get_current_time(),
            "modified": get_current_time(),
            "pattern_type": "stix",
            "pattern_version": "2.1",
            "valid_from": get_current_time()
        }
    ],
}

indicator

{'type': 'bundle',
 'id': 'bundle--e772e064-0c09-4f57-9302-58d5ce74b2c0',
 'spec_version': '2.1',
 'objects': [{'description': 'Detects SSRF attempts to fetch Google Cloud service account tokens from the metadata server.',
   'indicator_types': ['malicious-activity'],
   'kill_chain_phases': [{'kill_chain_name': 'mitre-attack',
     'phase_name': 'credential-access'}],
   'name': 'SSRF - Google Cloud Metadata Server Token Fetch',
   'pattern': "([artifact:message MATCHES '.*\\/computeMetadata\\/v1\\/instance\\/service-accounts\\/default\\/token.*'] FOLLOWEDBY [artifact:message MATCHES '.*metadata.google.internal.*'])",
   'type': 'indicator',
   'spec_version': '2.1',
   'id': 'indicator--05a20604-164d-4fae-8e11-17e6f9339775',
   'created': '2024-10-07T18:08:13.880346Z',
   'modified': '2024-10-07T18:08:13.880423Z',
   'pattern_type': 'stix',
   'pattern_version': '2.1',
   'valid_from': '2024-10-07T18:08:13.880447Z'}]}